# MuScripter Flash Training (Colab)

This notebook can automatically download and extract the official MusicNet dataset, then run the guarded MuScripter Flash training pipeline. Keep `WORKDIR` on Google Drive so checkpoints and resume state survive a Colab disconnect.

Default flow: **download MusicNet → train/resume → compare against the bundled Bach10 checkpoint → promotion quality gates**. No checkpoint is installed automatically unless you explicitly set `PROMOTE_TO`.


In [ ]:
%pip -q install --upgrade "git+https://github.com/sakusdev/muscriptor.git@main" scipy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Configuration

For the easiest first run, leave `AUTO_DOWNLOAD_MUSICNET = True` and run all cells. The official archive is about 11.1 GB. By default it is downloaded/extracted under `/content`, so it does not consume Google Drive storage. Set `CACHE_MUSICNET_ARCHIVE_ON_DRIVE = True` if you prefer the large archive to persist across Colab runtime resets.

`WORKDIR` stays on Drive and contains the durable model/checkpoint state. Bach10 and URMP are optional and can be added later.


In [ ]:
from pathlib import Path

AUTO_DOWNLOAD_MUSICNET = True
CACHE_MUSICNET_ARCHIVE_ON_DRIVE = False

# Optional extra datasets. Leave as None for a MusicNet-only first run.
BACH10 = None  # e.g. Path('/content/drive/MyDrive/datasets/Bach10_v1.1')
URMP = None    # e.g. Path('/content/drive/MyDrive/datasets/URMP')

# If AUTO_DOWNLOAD_MUSICNET=False, set this to an already extracted MusicNet root.
MUSICNET = None
MUSICNET_DATA_DIR = Path('/content/muscriptor-datasets/musicnet')
MUSICNET_CACHE_DIR = (
    Path('/content/drive/MyDrive/MuScripterFlash/dataset-cache')
    if CACHE_MUSICNET_ARCHIVE_ON_DRIVE
    else Path('/content/muscriptor-datasets/cache')
)

WORKDIR = Path('/content/drive/MyDrive/MuScripterFlash/full-run')
TARGET_STEPS = 3000
BATCH_SIZE = 32
EVAL_EVERY = 50
SAVE_EVERY = 50

# Optional: only set this when you want a passing candidate copied somewhere.
PROMOTE_TO = None  # e.g. Path('/content/drive/MyDrive/MuScripterFlash/promoted.pt')


## Download / reuse MusicNet

The helper reuses an existing extraction, resumes an interrupted `.part` download when the server supports HTTP Range, verifies the published MD5 checksum, and extracts with path-safety checks.


In [ ]:
from muscriptor.flash_dataset_fetch import ensure_musicnet

if AUTO_DOWNLOAD_MUSICNET:
    MUSICNET = ensure_musicnet(
        MUSICNET_DATA_DIR,
        cache_dir=MUSICNET_CACHE_DIR,
        keep_archive=CACHE_MUSICNET_ARCHIVE_ON_DRIVE,
    )
elif MUSICNET is not None:
    MUSICNET = Path(MUSICNET)

print('Bach10 :', BACH10)
print('URMP   :', URMP)
print('MusicNet:', MUSICNET)


## Train → compare → promotion gates

Re-running this cell with the same `WORKDIR` resumes automatically from `training-state.pt`. `TARGET_STEPS` is the final target step, not an additional number of steps.


In [ ]:
from muscriptor.flash_pipeline import run_pipeline

result = run_pipeline(
    workdir=WORKDIR,
    bach10=BACH10,
    urmp=URMP,
    musicnet=MUSICNET,
    steps=TARGET_STEPS,
    batch_size=BATCH_SIZE,
    eval_every=EVAL_EVERY,
    save_every=SAVE_EVERY,
    promote_to=PROMOTE_TO,
)
result


## Durable outputs

`WORKDIR` contains `model.pt`, `metrics.json`, `training-state.pt`, `run.json`, `comparison.json`, `promotion.json`, and `pipeline.json`. If Colab disconnects, reconnect and run the notebook again; training resumes from the saved state.

To intentionally restart from step 0, call `run_pipeline(..., fresh=True)`. The managed trainer preserves the previous state as a timestamped backup.

The MusicNet archive is deleted after successful extraction when `CACHE_MUSICNET_ARCHIVE_ON_DRIVE=False`. With Drive caching enabled, it is retained so a fresh Colab runtime can re-extract without re-downloading the 11.1 GB archive.
